## YOLO Pose Rule based approach to detect Cheating Copying vs normal

In [ ]:
!pip install ultralytics opencv-python -q

# =========================================
# 2️- IMPORTS
# =========================================
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict, deque

# =========================================
# 3️- LOAD MODELS
# =========================================
print("Loading models...")

detector = YOLO("yolov8n.pt")         # person detection + tracking
pose_model = YOLO("yolov8n-pose.pt") # pose estimation

print("Models loaded!")

# =========================================
# 4️- PARAMETERS
# =========================================
VIDEO_PATH = "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0133.MOV"  # on directory base CHANGE THIS
OUTPUT_PATH = "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/output_DSC_0133.MOV"

WINDOW_SIZE = 15
SCORE_THRESHOLD = 5

# =========================================
# 5️- STORAGE
# =========================================
student_history = defaultdict(lambda: deque(maxlen=WINDOW_SIZE))
student_scores = defaultdict(float)
tracker_map = {}

# =========================================
# 6️- SIGNAL FUNCTIONS
# =========================================

def get_center(points):
    return np.mean(points[:, :2], axis=0)

def detect_side_glance(kp):
    nose = kp[0]
    left_eye = kp[1]
    right_eye = kp[2]

    if nose[0] < left_eye[0] - 10:
        return 1
    if nose[0] > right_eye[0] + 10:
        return 1
    return 0

def detect_leaning(kp):
    left_shoulder = kp[5]
    right_shoulder = kp[6]

    diff = abs(left_shoulder[1] - right_shoulder[1])
    return 1 if diff > 20 else 0

def detect_hand_movement(prev_kp, kp):
    if prev_kp is None:
        return 0

    movement = 0
    for i in [9, 10]:  # wrists
        movement += np.linalg.norm(kp[i][:2] - prev_kp[i][:2])

    return 1 if movement > 40 else 0

def detect_reaching(kp):
    left_wrist = kp[9]
    right_wrist = kp[10]
    torso = get_center(kp[5:7])

    if np.linalg.norm(left_wrist[:2] - torso) > 120:
        return 1
    if np.linalg.norm(right_wrist[:2] - torso) > 120:
        return 1

    return 0

def detect_bad_posture(kp):
    nose = kp[0]
    hip = kp[11]

    return 1 if abs(nose[0] - hip[0]) > 50 else 0

# =========================================
# 7️- PROCESS VIDEO
# =========================================
def run(video_path, output_path):

    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (w, h)
    )

    student_counter = 0
    frame_no = 0

    print("Processing video...")

    while True:

        ret, frame = cap.read()
        if not ret:
            break

        results = detector.track(
            frame,
            persist=True,
            classes=[0],
            conf=0.4,
            verbose=False
        )

        if results[0].boxes.id is not None:

            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy().astype(int)

            for box, tid in zip(boxes, ids):

                # Assign student ID
                if tid not in tracker_map:
                    student_counter += 1
                    tracker_map[tid] = student_counter

                sid = tracker_map[tid]

                x1, y1, x2, y2 = map(int, box)

                # Clamp
                x1 = max(0, x1); y1 = max(0, y1)
                x2 = min(w, x2); y2 = min(h, y2)

                crop = frame[y1:y2, x1:x2]

                if crop.size == 0:
                    continue

                # Pose detection
                pose = pose_model(crop, verbose=False)

                if len(pose[0].keypoints.data) == 0:
                    continue

                kp = pose[0].keypoints.data[0].cpu().numpy()

                prev_kp = student_history[sid][-1] if len(student_history[sid])>0 else None

                # ===== SIGNALS =====
                s1 = detect_side_glance(kp)
                s2 = detect_leaning(kp)
                s3 = detect_hand_movement(prev_kp, kp)
                s4 = detect_reaching(kp)
                s5 = detect_bad_posture(kp)

                score = s1 + s2 + s3 + s4 + s5

                # Add decay (IMPORTANT)
                student_scores[sid] *= 0.9
                student_scores[sid] += score

                student_history[sid].append(kp)

                # ===== DECISION =====
                if student_scores[sid] > SCORE_THRESHOLD:
                    color = (0, 0, 255)
                    label = f"CHEATING {student_scores[sid]:.1f}"
                else:
                    color = (0, 255, 0)
                    label = f"Normal {student_scores[sid]:.1f}"

                # Draw
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)

                cv2.putText(
                    frame,
                    f"ID:{sid} {label}",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    color,
                    2
                )

        out.write(frame)

        if frame_no % 100 == 0:
            print("Frame:", frame_no)

        frame_no += 1

    cap.release()
    out.release()

    print("✅ Done! Output saved at:", output_path)


# =========================================
# 8️- RUN
# =========================================
run(VIDEO_PATH, OUTPUT_PATH)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading models...
Models loaded!
Processing video...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 226ms
Prepared 1 package in 53ms
Installed 1 package in 5ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Frame: 0
Frame: 100
Frame: 200
Frame: 300
Frame: 400
Frame: 500
Frame: 600
✅ Done! Output saved at: /content/drive/MyDrive/ourDataset/dataset_f5_hall-2/output_DSC_0133.MOV


In [ ]:
# =========================================
# 1️⃣ INSTALL
# =========================================
!pip install ultralytics opencv-python -q

# =========================================
# 2️⃣ IMPORTS
# =========================================
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict, deque

# =========================================
# 3️⃣ LOAD MODELS
# =========================================
detector = YOLO("yolov8n.pt")
pose_model = YOLO("yolov8n-pose.pt")

# =========================================
# 4️⃣ PARAMETERS
# =========================================
VIDEO_PATH = "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0125.MOV"   # 🔥 change
OUTPUT_PATH = "/content/drive/MyDrive/ourDataset/video_annotate/cropped_videos_new/output_DSC_0125.MOV"

WINDOW_SIZE = 15
SCORE_THRESHOLD = 5

# =========================================
# 5️⃣ STORAGE
# =========================================
student_history = defaultdict(lambda: deque(maxlen=WINDOW_SIZE))
student_scores = defaultdict(float)
tracker_map = {}

# =========================================
# 6️⃣ SIGNAL FUNCTIONS
# =========================================
def get_center(points):
    return np.mean(points[:, :2], axis=0)

def side_glance(kp):
    nose, le, re = kp[0], kp[1], kp[2]
    return 1 if (nose[0] < le[0]-10 or nose[0] > re[0]+10) else 0

def leaning(kp):
    return 1 if abs(kp[5][1] - kp[6][1]) > 20 else 0

def hand_move(prev, kp):
    if prev is None: return 0
    mv = sum(np.linalg.norm(kp[i][:2]-prev[i][:2]) for i in [9,10])
    return 1 if mv > 40 else 0

def reaching(kp):
    torso = get_center(kp[5:7])
    return 1 if (
        np.linalg.norm(kp[9][:2]-torso) > 120 or
        np.linalg.norm(kp[10][:2]-torso) > 120
    ) else 0

def posture(kp):
    return 1 if abs(kp[0][0]-kp[11][0]) > 50 else 0

# =========================================
# 7️⃣ DRAW POSE LANDMARKS
# =========================================
def draw_landmarks(img, kp, offset=(0,0)):
    for x,y,conf in kp:
        if conf > 0.3:
            cv2.circle(img, (int(x)+offset[0], int(y)+offset[1]), 3, (255,255,0), -1)

# =========================================
# 8️⃣ DRAW DASHBOARD (RIGHT PANEL)
# =========================================
def draw_dashboard(frame, student_scores):

    h, w, _ = frame.shape
    panel_w = 300

    # Background panel
    cv2.rectangle(frame, (w-panel_w,0), (w,h), (30,30,30), -1)

    y = 40

    for sid, score in sorted(student_scores.items()):

        # Normalize score
        norm = min(score / SCORE_THRESHOLD, 1.0)

        # Bar size
        bar_w = int(200 * norm)

        # Color
        if score > SCORE_THRESHOLD:
            color = (0,0,255)  # RED
            status = "CHEATING"
        else:
            color = (0,255,0)  # GREEN
            status = "NORMAL"

        # Text
        cv2.putText(frame, f"ID {sid}", (w-panel_w+10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

        # Bar background
        cv2.rectangle(frame, (w-panel_w+10, y+10),
                      (w-panel_w+210, y+30), (80,80,80), -1)

        # Bar fill
        cv2.rectangle(frame, (w-panel_w+10, y+10),
                      (w-panel_w+10+bar_w, y+30), color, -1)

        # Status text
        cv2.putText(frame, status, (w-panel_w+10, y+50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        y += 70

# =========================================
# 9️⃣ MAIN PROCESS
# =========================================
def run(video_path, output_path):

    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (w, h)
    )

    student_counter = 0
    frame_no = 0

    print("Processing...")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = detector.track(frame, persist=True, classes=[0], conf=0.4, verbose=False)

        if results[0].boxes.id is not None:

            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy().astype(int)

            for box, tid in zip(boxes, ids):

                if tid not in tracker_map:
                    student_counter += 1
                    tracker_map[tid] = student_counter

                sid = tracker_map[tid]

                x1,y1,x2,y2 = map(int, box)
                crop = frame[y1:y2, x1:x2]

                if crop.size == 0:
                    continue

                pose = pose_model(crop, verbose=False)

                if len(pose[0].keypoints.data) == 0:
                    continue

                kp = pose[0].keypoints.data[0].cpu().numpy()

                prev = student_history[sid][-1] if len(student_history[sid])>0 else None

                # Signals
                score = (
                    side_glance(kp) +
                    leaning(kp) +
                    hand_move(prev,kp) +
                    reaching(kp) +
                    posture(kp)
                )

                # Decay
                student_scores[sid] *= 0.9
                student_scores[sid] += score

                student_history[sid].append(kp)

                # Decision
                if student_scores[sid] > SCORE_THRESHOLD:
                    color = (0,0,255)
                else:
                    color = (0,255,0)

                # Box
                cv2.rectangle(frame,(x1,y1),(x2,y2),color,3)

                # Label
                cv2.putText(frame,
                            f"ID:{sid} {student_scores[sid]:.1f}",
                            (x1,y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6,color,2)

                # Landmarks (IMPORTANT OFFSET)
                draw_landmarks(frame, kp, offset=(x1,y1))

        # Draw dashboard
        draw_dashboard(frame, student_scores)

        out.write(frame)

        if frame_no % 100 == 0:
            print("Frame:", frame_no)

        frame_no += 1

    cap.release()
    out.release()

    print("✅ DONE:", output_path)


# =========================================
# 🔟 RUN
# =========================================
run(VIDEO_PATH, OUTPUT_PATH)

Processing...
Frame: 0
Frame: 100
Frame: 200
Frame: 300
Frame: 400
Frame: 500
Frame: 600
✅ DONE: /content/drive/MyDrive/ourDataset/video_annotate/cropped_videos_new/output_DSC_0125.MOV


# Selective landmarks
# solve false postive issue i.e: normal call cheating

In [ ]:
# 1️⃣ INSTALL
# =========================================
!pip install ultralytics opencv-python -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.5 MB/s eta 0:00:00


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict, deque

# =========================================
# MODELS
# =========================================
detector = YOLO("yolov8n.pt")
pose_model = YOLO("yolov8n-pose.pt")

# =========================================
# SETTINGS
# =========================================
VIDEO_PATH = "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0129.MOV"   # 🔥 change
OUTPUT_PATH = "/content/drive/MyDrive/ourDataset/video_annotate/cropped_videos_new/output_vii_DSC_0129.MOV"


CLIP_MEMORY = 20
SCORE_THRESHOLD = 6

# =========================================
# STORAGE
# =========================================
history = defaultdict(lambda: deque(maxlen=CLIP_MEMORY))
scores = defaultdict(float)
tracker_map = {}

streaks = defaultdict(lambda: {
    "glance":0,
    "lean":0,
    "reach":0
})

# =========================================
# RULES (UPDATED)
# =========================================

# 🔥 BETTER SIDE GLANCE (EAR + EYE BASED)
def side_glance(kp):

    nose = kp[0]
    leye = kp[1]
    reye = kp[2]
    lear = kp[3]
    rear = kp[4]

    # If one ear disappears → side view
    if lear[2] < 0.3 or rear[2] < 0.3:
        return 1

    # Face shift from center
    face_center = (leye[0] + reye[0]) / 2
    if abs(nose[0] - face_center) > 25:
        return 1

    return 0


# 🔥 FIXED LEANING (IGNORE SMALL TILT)
def leaning(kp):

    ls = kp[5]
    rs = kp[6]

    # small tilt = normal
    if abs(ls[1] - rs[1]) < 35:
        return 0

    return 1


# 🔥 REACHING
def reaching(kp):

    lw = kp[9]
    rw = kp[10]

    torso = (kp[5][:2] + kp[6][:2]) / 2

    if np.linalg.norm(lw[:2] - torso) > 150:
        return 1
    if np.linalg.norm(rw[:2] - torso) > 150:
        return 1

    return 0


# 🔥 HAND MOVEMENT (IGNORE WRITING)
def hand_movement(prev, kp):

    if prev is None:
        return 0

    lw = kp[9]; rw = kp[10]
    plw = prev[9]; prw = prev[10]

    move = np.linalg.norm(lw[:2]-plw[:2]) + np.linalg.norm(rw[:2]-prw[:2])

    # normal writing → ignore
    if move < 30:
        return 0

    # strong abnormal motion
    if move > 90:
        return 1

    return 0


# =========================================
# DRAW FUNCTIONS
# =========================================
# Removed elbow landmarks (7,8) and their connections.
# New connections go directly from shoulder to wrist (5->9, 6->10)
SKELETON = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),
    (5,9),
    (6,10),
    (5,11),(6,12),(11,12)
]

def draw_pose(frame, kp, offset):

    ox, oy = offset

    for i,j in SKELETON:
        if kp[i][2]>0.3 and kp[j][2]>0.3:
            x1,y1 = int(kp[i][0])+ox, int(kp[i][1])+oy
            x2,y2 = int(kp[j][0])+ox, int(kp[j][1])+oy
            cv2.line(frame,(x1,y1),(x2,y2),(255,0,0),2)

    for i in range(len(kp)):
        # Exclude elbow landmarks (7 and 8) from drawing circles
        if i not in [7, 8]:
            x,y,c = kp[i]
            if c>0.3:
                cv2.circle(frame,(int(x)+ox,int(y)+oy),4,(0,255,255),-1)


def draw_dashboard(frame, scores):

    h,w,_ = frame.shape
    panel = 260

    # Draw transparent background for the panel
    overlay = frame.copy()
    cv2.rectangle(overlay,(w-panel,0),(w,h),(30,30,30),-1) # Dark grey background
    alpha = 0.6 # Transparency factor (0.0 - fully transparent, 1.0 - fully opaque)
    frame[:] = cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

    y = 40

    for sid,score in scores.items():

        norm = min(score/SCORE_THRESHOLD,1)
        bar = int(160*norm)

        if score > SCORE_THRESHOLD:
            color = (0,0,255)
            status = "CHEAT"
        else:
            color = (0,255,0)
            status = "NORMAL"

        cv2.putText(frame,f"ID {sid}",(w-panel+10,y),
                    cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)

        cv2.rectangle(frame,(w-panel+10,y+10),
                      (w-panel+170,y+30),(80,80,80),-1)

        cv2.rectangle(frame,(w-panel+10,y+10),
                      (w-panel+10+bar,y+30),color,-1)

        cv2.putText(frame,status,(w-panel+10,y+50),
                    cv2.FONT_HERSHEY_SIMPLEX,0.6,color,2)

        y += 70


# =========================================
# MAIN
# =========================================
def run():

    cap = cv2.VideoCapture(VIDEO_PATH)

    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(3))
    h = int(cap.get(4))

    out = cv2.VideoWriter(OUTPUT_PATH,
                          cv2.VideoWriter_fourcc(*'mp4v'),
                          fps,(w,h))

    sid_counter = 0

    while True:

        ret,frame = cap.read()
        if not ret:
            break

        results = detector.track(frame,persist=True,classes=[0],conf=0.4,verbose=False)

        if results[0].boxes.id is not None:

            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy().astype(int)

            for box,tid in zip(boxes,ids):

                if tid not in tracker_map:
                    sid_counter += 1
                    tracker_map[tid] = sid_counter

                sid = tracker_map[tid]

                x1,y1,x2,y2 = map(int,box)
                crop = frame[y1:y2,x1:x2]

                if crop.size == 0:
                    continue

                pose = pose_model(crop,verbose=False)

                if len(pose[0].keypoints.data)==0:
                    continue

                kp = pose[0].keypoints.data[0].cpu().numpy()

                prev = history[sid][-1] if len(history[sid])>0 else None

                # ===== RULES =====
                g = side_glance(kp)
                l = leaning(kp)
                r = reaching(kp)
                hm = hand_movement(prev,kp)

                # ===== STREAK =====
                streaks[sid]["glance"] = streaks[sid]["glance"]+1 if g else 0
                streaks[sid]["lean"] = streaks[sid]["lean"]+1 if l else 0
                streaks[sid]["reach"] = streaks[sid]["reach"]+1 if r else 0

                score = 0

                if streaks[sid]["glance"] > 20:
                    score += 2

                if streaks[sid]["lean"] > 25:  # 🔥 more strict
                    score += 1

                if streaks[sid]["reach"] > 10:
                    score += 2

                score += hm

                # ===== DECAY =====
                scores[sid] *= 0.85
                scores[sid] += score
                scores[sid] = min(scores[sid],10)

                history[sid].append(kp)

                color = (0,0,255) if scores[sid]>SCORE_THRESHOLD else (0,255,0)

                cv2.rectangle(frame,(x1,y1),(x2,y2),color,3)

                cv2.putText(frame,
                            f"ID:{sid} {scores[sid]:.1f}",
                            (x1,y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.9,color,3)

                draw_pose(frame,kp,(x1,y1))

        draw_dashboard(frame,scores)

        out.write(frame)

    cap.release()
    out.release()

    print("✅ FINAL FIXED VERSION SAVED")


# =========================================
# RUN
# =========================================
run()


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 314ms
Prepared 1 package in 73ms
Installed 1 package in 3ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

✅ FINAL FIXED VERSION SAVED


# Person tracking issue resolve

In [ ]:
# 1️⃣ INSTALL
# =========================================
!pip install ultralytics opencv-python -q

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict, deque
# from google.colab.patches import cv2_imshow

# =========================================
# MODELS
# =========================================
detector = YOLO("yolov8n.pt")
pose_model = YOLO("yolov8n-pose.pt")

# =========================================
# SETTINGS
# =========================================
# 🔥 0 = WEBCAM (REAL-TIME) | or give video path
VIDEO_PATH = "/content/drive/MyDrive/ourDataset/dataset_f5_hall-2/DSC_0129.MOV"   # 🔥 change
OUTPUT_PATH = "/content/drive/MyDrive/ourDataset/video_annotate/cropped_videos_new/output_trackii_DSC_0129.MOV"


CLIP_MEMORY = 20
SCORE_THRESHOLD = 6

# =========================================
# STORAGE
# =========================================
history = defaultdict(lambda: deque(maxlen=CLIP_MEMORY))
scores = defaultdict(float)
tracker_map = {}
student_features = {}
new_track_buffer = defaultdict(int)

streaks = defaultdict(lambda: {
    "glance":0,
    "lean":0,
    "reach":0
})

# =========================================
# RE-ID
# =========================================
def extract_feature(crop):
    crop = cv2.resize(crop,(64,128))
    hist = cv2.calcHist([crop],[0,1,2],None,[8,8,8],[0,256]*3)
    hist = cv2.normalize(hist,hist).flatten()
    return hist

def match_student(feature):
    best_id = None
    best_score = 999

    for sid,feat in student_features.items():
        dist = np.linalg.norm(feature - feat)
        if dist < best_score:
            best_score = dist
            best_id = sid

    if best_score < 0.6:
        return best_id

    return None

# =========================================
# RULES
# =========================================
def side_glance(kp):
    nose, leye, reye, lear, rear = kp[0], kp[1], kp[2], kp[3], kp[4]

    if lear[2] < 0.3 or rear[2] < 0.3:
        return 1

    face_center = (leye[0] + reye[0]) / 2
    if abs(nose[0] - face_center) > 25:
        return 1

    return 0

def leaning(kp):
    ls, rs = kp[5], kp[6]
    if abs(ls[1] - rs[1]) < 35:
        return 0
    return 1

def reaching(kp):
    lw, rw = kp[9], kp[10]
    torso = (kp[5][:2] + kp[6][:2]) / 2

    if np.linalg.norm(lw[:2] - torso) > 150:
        return 1
    if np.linalg.norm(rw[:2] - torso) > 150:
        return 1

    return 0

def hand_movement(prev, kp):
    if prev is None:
        return 0

    lw, rw = kp[9], kp[10]
    plw, prw = prev[9], prev[10]

    move = np.linalg.norm(lw[:2]-plw[:2]) + np.linalg.norm(rw[:2]-prw[:2])

    if move < 30:
        return 0
    if move > 90:
        return 1

    return 0

# =========================================
# SKELETON
# =========================================
SKELETON = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),
    (5,9),(6,10),
    (5,11),(6,12),(11,12)
]

def draw_pose(frame, kp, offset):
    ox, oy = offset

    for i,j in SKELETON:
        if kp[i][2]>0.3 and kp[j][2]>0.3:
            x1,y1 = int(kp[i][0])+ox, int(kp[i][1])+oy
            x2,y2 = int(kp[j][0])+ox, int(kp[j][1])+oy
            cv2.line(frame,(x1,y1),(x2,y2),(255,0,0),2)

    for i in range(len(kp)):
        if i not in [7,8]:
            x,y,c = kp[i]
            if c>0.3:
                cv2.circle(frame,(int(x)+ox,int(y)+oy),4,(0,255,255),-1)

# =========================================
# DASHBOARD
# =========================================
def draw_dashboard(frame, scores):
    h,w,_ = frame.shape
    panel = 260

    overlay = frame.copy()
    cv2.rectangle(overlay,(w-panel,0),(w,h),(30,30,30),-1)
    frame[:] = cv2.addWeighted(overlay,0.6,frame,0.4,0)

    y = 40

    for sid in sorted(scores.keys()):
        score = scores[sid]

        norm = min(score/SCORE_THRESHOLD,1)
        bar = int(160*norm)

        if score > SCORE_THRESHOLD:
            color = (0,0,255)
            status = "CHEAT"
        else:
            color = (0,255,0)
            status = "NORMAL"

        cv2.putText(frame,f"ID {sid}",(w-panel+10,y),
                    cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)

        cv2.rectangle(frame,(w-panel+10,y+10),
                      (w-panel+170,y+30),(80,80,80),-1)

        cv2.rectangle(frame,(w-panel+10,y+10),
                      (w-panel+10+bar,y+30),color,-1)

        cv2.putText(frame,status,(w-panel+10,y+50),
                    cv2.FONT_HERSHEY_SIMPLEX,0.6,color,2)

        y += 70

# =========================================
# MAIN
# =========================================
# def run():

#     cap = cv2.VideoCapture(VIDEO_PATH)

#     if not cap.isOpened():
#         print("Error opening video")
#         return

#     fps = cap.get(cv2.CAP_PROP_FPS)
#     if fps == 0: fps = 25

#     w = int(cap.get(3))
#     h = int(cap.get(4))

#     out = cv2.VideoWriter(OUTPUT_PATH,
#                           cv2.VideoWriter_fourcc(*'mp4v'),
#                           fps,(w,h))

#     sid_counter = 0

#     # -------

#     frame_no = 0

#     while True:
#         ret,frame = cap.read()
#         if not ret:
#             break

#         results = detector.track(frame,persist=True,classes=[0],conf=0.4,verbose=False)

#         if results[0].boxes.id is not None:

#             boxes = results[0].boxes.xyxy.cpu().numpy()
#             ids = results[0].boxes.id.cpu().numpy().astype(int)

#             for box,tid in zip(boxes,ids):

#                 x1,y1,x2,y2 = map(int,box)
#                 crop = frame[y1:y2,x1:x2]

#                 if crop.size == 0:
#                     continue

#                 # 🔥 DELAYED ID ASSIGNMENT
#                 if tid not in tracker_map:

#                     new_track_buffer[tid] += 1

#                     if new_track_buffer[tid] < 5:
#                         continue

#                     feature = extract_feature(crop)
#                     matched_id = match_student(feature)

#                     if matched_id is not None:
#                         sid = matched_id
#                     else:
#                         sid_counter += 1
#                         sid = sid_counter
#                         student_features[sid] = feature

#                     tracker_map[tid] = sid

#                 sid = tracker_map[tid]

#                 pose = pose_model(crop,verbose=False)

#                 if len(pose[0].keypoints.data)==0:
#                     continue

#                 kp = pose[0].keypoints.data[0].cpu().numpy()
#                 prev = history[sid][-1] if len(history[sid])>0 else None

#                 g = side_glance(kp)
#                 l = leaning(kp)
#                 r = reaching(kp)
#                 hm = hand_movement(prev,kp)

#                 streaks[sid]["glance"] = streaks[sid]["glance"]+1 if g else 0
#                 streaks[sid]["lean"] = streaks[sid]["lean"]+1 if l else 0
#                 streaks[sid]["reach"] = streaks[sid]["reach"]+1 if r else 0

#                 score = 0

#                 if streaks[sid]["glance"] > 25:
#                     score += 2
#                 if streaks[sid]["lean"] > 30:
#                     score += 1
#                 if streaks[sid]["reach"] > 12:
#                     score += 2

#                 score += hm


#                 scores[sid] *= 0.85
#                 scores[sid] += score
#                 scores[sid] = min(scores[sid],10)

#                 history[sid].append(kp)

#                 color = (0,0,255) if scores[sid]>SCORE_THRESHOLD else (0,255,0)

#                 cv2.rectangle(frame,(x1,y1),(x2,y2),color,3)

#                 cv2.putText(frame,
#                             f"ID:{sid} {scores[sid]:.1f}",
#                             (x1,y1-10),
#                             cv2.FONT_HERSHEY_SIMPLEX,
#                             0.9,color,3)

#                 draw_pose(frame,kp,(x1,y1))

#         draw_dashboard(frame,scores)

#         out.write(frame)

#         # if frame_no % 10 == 0:
#         #    cv2_imshow(frame)

#         frame_no += 1
#         if cv2.waitKey(1) & 0xFF == 27:
#             break

#     cap.release()
#     out.release()
#     cv2.destroyAllWindows()

#     print("✅ Done. Output saved:", OUTPUT_PATH)
def run():

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        print("Error opening video")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0: fps = 25

    w = int(cap.get(3))
    h = int(cap.get(4))

    out = cv2.VideoWriter(OUTPUT_PATH,
                          cv2.VideoWriter_fourcc(*'mp4v'),
                          fps,(w,h))

    sid_counter = 0

    while True:
        ret,frame = cap.read()
        if not ret:
            break

        results = detector.track(frame, persist=True, classes=[0], conf=0.4, verbose=False)

        if results[0].boxes.id is not None:

            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy().astype(int)

            for box,tid in zip(boxes,ids):

                x1,y1,x2,y2 = map(int,box)
                crop = frame[y1:y2,x1:x2]

                if crop.size == 0:
                    continue

                # ===============================
                # 🔥 FIXED ID LOGIC
                # ===============================
                if tid in tracker_map:
                    sid = tracker_map[tid]

                else:
                    # Try re-identification ONLY if new ID appears
                    feature = extract_feature(crop)
                    matched_id = match_student(feature)

                    if matched_id is not None:
                        sid = matched_id
                    else:
                        sid_counter += 1
                        sid = sid_counter
                        student_features[sid] = feature

                    tracker_map[tid] = sid

                # ===============================
                # POSE
                # ===============================
                pose = pose_model(crop, verbose=False)

                if len(pose[0].keypoints.data) == 0:
                    continue

                kp = pose[0].keypoints.data[0].cpu().numpy()

                prev = history[sid][-1] if len(history[sid]) > 0 else None

                # ===============================
                # SAME RULES (UNCHANGED)
                # ===============================
                g = side_glance(kp)
                l = leaning(kp)
                r = reaching(kp)
                hm = hand_movement(prev, kp)

                streaks[sid]["glance"] = streaks[sid]["glance"]+1 if g else 0
                streaks[sid]["lean"] = streaks[sid]["lean"]+1 if l else 0
                streaks[sid]["reach"] = streaks[sid]["reach"]+1 if r else 0

                score = 0

                if streaks[sid]["glance"] > 20:
                    score += 2

                if streaks[sid]["lean"] > 25:
                    score += 1

                if streaks[sid]["reach"] > 10:
                    score += 2

                score += hm

                # ===============================
                # DECAY (UNCHANGED)
                # ===============================
                scores[sid] *= 0.85
                scores[sid] += score
                scores[sid] = min(scores[sid],10)

                history[sid].append(kp)

                # ===============================
                # DRAW
                # ===============================
                color = (0,0,255) if scores[sid] > SCORE_THRESHOLD else (0,255,0)

                cv2.rectangle(frame,(x1,y1),(x2,y2),color,3)

                cv2.putText(frame,
                            f"ID:{sid} {scores[sid]:.1f}",
                            (x1,y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.9,color,3)

                draw_pose(frame,kp,(x1,y1))

        draw_dashboard(frame,scores)

        out.write(frame)

    cap.release()
    out.release()

    print("✅ FINAL FIXED (ID STABLE + RULES SAME)")

# =========================================
run()

✅ FINAL FIXED (ID STABLE + RULES SAME)
